# Week 5 — Day 2: DBSCAN & Hierarchical Clustering

Day 1 introduced K-Means and showed how a centroid-based method can separate compact groups.  
This notebook continues the clustering work with two alternatives:

- **DBSCAN**, which groups dense connected regions and can leave sparse points as noise.
- **Agglomerative Hierarchical Clustering**, which starts with one cluster per point and repeatedly merges the closest clusters.

The main goal is not only to run the algorithms, but to compare what each method assumes about the shape of the data and decide which one fits this dataset best.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import make_moons
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

## 2. Dataset

I use a two-moons dataset with a small number of additional outlier points.  
This shape is useful for comparing clustering methods because the two groups are curved rather than circular.

K-Means can still split the space, but its assignments are based on the nearest centroid. DBSCAN instead follows dense connected regions, so this dataset makes the difference between the two approaches visible.

In [ ]:
X_moons, _ = make_moons(
    n_samples=400,
    noise=0.07,
    random_state=42,
)

rng = np.random.default_rng(42)
outliers = rng.uniform(
    low=[-1.5, -1.0],
    high=[2.5, 1.5],
    size=(12, 2),
)

X = np.vstack([X_moons, outliers])

df = pd.DataFrame(X, columns=["x1", "x2"])
df.head()

In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Features: {df.shape[1]}")
print(f"Missing values: {df.isna().sum().sum()}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["x1"], df["x2"], s=28)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Original Data")
plt.show()

The plot contains two curved dense regions and a few isolated points.  
This is not the geometry K-Means handles most naturally because one centroid cannot represent the shape of an entire moon very well.

The isolated points are also important: K-Means must assign every sample to a cluster, while DBSCAN can mark points that do not belong to a dense region as noise.

## 3. Scaling Before Distance-Based Clustering

All three methods in this notebook depend on distances. If one feature had a much larger numeric range than another, it could dominate the distance calculation.

I standardize the two features before clustering so that the comparison is based on the geometry of the data rather than the original feature scales.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

pd.DataFrame(X_scaled, columns=["x1_scaled", "x2_scaled"]).describe().round(3)

## 4. K-Means Baseline

I first keep K-Means as a baseline with `k=2`.

K-Means assigns each point to its nearest centroid. This creates convex regions around the centroids, so it is usually strongest when clusters are compact and roughly spherical. It also has no noise label, so every point must be assigned somewhere.

In [ ]:
kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10,
)

kmeans_labels = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(8, 5))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=kmeans_labels, s=28)
plt.scatter(
    kmeans.cluster_centers_[:, 0],
    kmeans.cluster_centers_[:, 1],
    marker="X",
    s=180,
)
plt.xlabel("x1 (scaled)")
plt.ylabel("x2 (scaled)")
plt.title("K-Means Baseline")
plt.show()

The split is based on the two centroid locations rather than the curved structure of the moons.  
This is the main limitation I want to compare against DBSCAN: the natural groups can be connected and non-convex even when a centroid-based partition cuts across them.

## 5. DBSCAN Intuition

DBSCAN defines a cluster as a **dense connected region**.

Two parameters define what "dense" means:

- `eps`: the maximum neighborhood radius.
- `min_samples`: the minimum number of samples required inside that neighborhood for a point to be a core point.

A **core point** has enough nearby samples to satisfy the density condition.  
A **border point** belongs to the neighborhood of a core point but does not have enough nearby samples to expand the cluster itself.  
A point that is not reached from any dense region is labeled as **noise** (`-1`).

The cluster grows through connected core points. Border points can be included, but they do not continue the expansion.

## 6. Choosing DBSCAN Parameters

I do not want to choose `eps` only because one value looks good on the final plot.

I start with `min_samples=5`, then inspect the distance from every point to its 5th nearest neighbor.  
When these distances are sorted, the upper bend of the curve gives a useful region to inspect for `eps`.

This is a starting point, not an automatic proof of the best parameter.

In [ ]:
min_samples = 5

neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors.fit(X_scaled)

distances, _ = neighbors.kneighbors(X_scaled)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(8, 5))
plt.plot(k_distances)
plt.xlabel("Points sorted by 5th-neighbor distance")
plt.ylabel("5th-neighbor distance")
plt.title("K-Distance Plot")
plt.show()

The curve stays relatively low for most samples and rises near the sparse points.  
I test a small range around that transition instead of searching a very large parameter grid.

The effect of the parameters is important:

- smaller `eps` → fewer neighbors, more fragmented clusters, more noise;
- larger `eps` → more connected points, less noise, but separate clusters can merge;
- larger `min_samples` → a stricter density requirement and usually more noise;
- smaller `min_samples` → an easier density requirement, but weak local patterns may become clusters.

In [ ]:
eps_values = [0.18, 0.20, 0.22, 0.25, 0.30]
dbscan_trials = []

for eps in eps_values:
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_scaled)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int(np.sum(labels == -1))

    mask = labels != -1
    if n_clusters >= 2 and np.unique(labels[mask]).size >= 2:
        silhouette = silhouette_score(X_scaled[mask], labels[mask])
    else:
        silhouette = np.nan

    dbscan_trials.append(
        {
            "eps": eps,
            "clusters": n_clusters,
            "noise_points": n_noise,
            "silhouette_excluding_noise": silhouette,
        }
    )

dbscan_results = pd.DataFrame(dbscan_trials)
dbscan_results.round(4)

The parameter table is more useful than selecting a value from one plot alone.  
I look for a stable region where the two expected dense structures remain separate and the noise count does not change sharply with a tiny parameter change.

For this dataset, `eps=0.25` with `min_samples=5` gives two clusters and keeps the sparse points outside the dense regions.

In [ ]:
dbscan = DBSCAN(
    eps=0.25,
    min_samples=min_samples,
)

dbscan_labels = dbscan.fit_predict(X_scaled)

n_dbscan_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_dbscan_noise = int(np.sum(dbscan_labels == -1))

print(f"DBSCAN clusters: {n_dbscan_clusters}")
print(f"Noise points: {n_dbscan_noise}")

In [ ]:
core_mask = np.zeros(len(X_scaled), dtype=bool)
core_mask[dbscan.core_sample_indices_] = True

noise_mask = dbscan_labels == -1
border_mask = ~(core_mask | noise_mask)

print(f"Core points: {core_mask.sum()}")
print(f"Border points: {border_mask.sum()}")
print(f"Noise points: {noise_mask.sum()}")

In [ ]:
plt.figure(figsize=(8, 5))

for label in sorted(set(dbscan_labels)):
    mask = dbscan_labels == label
    name = "Noise" if label == -1 else f"Cluster {label}"
    plt.scatter(
        X_scaled[mask, 0],
        X_scaled[mask, 1],
        s=30,
        label=name,
    )

plt.xlabel("x1 (scaled)")
plt.ylabel("x2 (scaled)")
plt.title("DBSCAN Clustering")
plt.legend()
plt.show()

DBSCAN follows the curved dense regions instead of separating the data around two centroids.  
It also leaves sparse samples as noise instead of forcing them into one of the clusters.

This result is only meaningful together with the parameter check above. If `eps` were much smaller, the moons would fragment. If it were too large, connected neighborhoods could merge clusters that should remain separate.

## 7. Agglomerative Hierarchical Clustering

Agglomerative clustering uses a bottom-up process:

1. every sample starts as its own cluster;
2. the two closest clusters are merged;
3. distances between the new cluster and the remaining clusters are updated;
4. the process repeats until all samples belong to one hierarchy.

The important detail is step 3. After two points are merged, the new cluster is not a new physical point with its own Euclidean coordinates.  
A **linkage rule** defines the distance between clusters.

For clusters \(A\) and \(B\):

- **Single linkage**: shortest pairwise distance between the two clusters;
- **Complete linkage**: longest pairwise distance;
- **Average linkage**: average of all cross-cluster pairwise distances;
- **Ward linkage**: chooses the merge that causes the smallest increase in within-cluster variance.

For example, if `P3` and `P6` are merged into `{P3, P6}`, single linkage defines its distance to `P1` as:

\[
d(\{P3,P6\},P1)=\min(d(P3,P1), d(P6,P1))
\]

So the distance matrix shrinks by replacing the two old entries with one cluster entry, and only the distances involving that new cluster need to be recalculated.

## 8. Dendrogram with Ward Linkage

I use Ward linkage because it is the linkage method used in this week's curriculum example.

The dendrogram records the sequence of merges. The vertical height of a merge represents the linkage distance at which two branches are joined.

A horizontal cut through the dendrogram converts the hierarchy into a flat set of clusters.

In [ ]:
Z = linkage(X_scaled, method="ward")

plt.figure(figsize=(12, 6))
dendrogram(
    Z,
    truncate_mode="lastp",
    p=35,
    show_leaf_counts=True,
)
plt.xlabel("Cluster / sample group")
plt.ylabel("Ward linkage distance")
plt.title("Agglomerative Clustering Dendrogram")
plt.show()

The full hierarchy contains one merge sequence for all samples, so I use a truncated dendrogram to keep the upper structure readable.

To obtain two clusters, I choose a cut height between the final merge and the merge immediately before it. This is a reproducible way to cut this specific hierarchy rather than selecting a height only by eye.

In [ ]:
previous_merge_height = Z[-2, 2]
final_merge_height = Z[-1, 2]
cut_height = (previous_merge_height + final_merge_height) / 2

hierarchical_labels = fcluster(
    Z,
    t=cut_height,
    criterion="distance",
)

n_hierarchical_clusters = len(np.unique(hierarchical_labels))

print(f"Previous merge height: {previous_merge_height:.3f}")
print(f"Final merge height: {final_merge_height:.3f}")
print(f"Chosen cut height: {cut_height:.3f}")
print(f"Clusters after cut: {n_hierarchical_clusters}")

In [ ]:
plt.figure(figsize=(12, 6))
dendrogram(
    Z,
    truncate_mode="lastp",
    p=35,
    show_leaf_counts=True,
)
plt.axhline(cut_height, linestyle="--", label=f"Cut = {cut_height:.2f}")
plt.xlabel("Cluster / sample group")
plt.ylabel("Ward linkage distance")
plt.title("Dendrogram with Selected Cut Height")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    X_scaled[:, 0],
    X_scaled[:, 1],
    c=hierarchical_labels,
    s=28,
)
plt.xlabel("x1 (scaled)")
plt.ylabel("x2 (scaled)")
plt.title("Hierarchical Clustering — Ward Linkage")
plt.show()

Ward linkage prefers compact groups because each merge is chosen to keep within-cluster variance small.  
That makes it useful for many hierarchical clustering problems, but it does not naturally follow the curved moon geometry as well as density connectivity does here.

The dendrogram is still useful because it exposes the nested merge structure instead of returning only one final partition.

## 9. Comparing the Three Methods

For the comparison, I report the number of clusters and the silhouette score.

For DBSCAN, I calculate silhouette only on non-noise samples because `-1` represents points that the algorithm intentionally leaves outside the clusters.

Silhouette is useful as one internal measure, but I do not use it as the only decision rule. The geometry of the clusters and the behavior of noise points are also part of the method choice.

In [ ]:
kmeans_silhouette = silhouette_score(X_scaled, kmeans_labels)

dbscan_non_noise = dbscan_labels != -1
dbscan_silhouette = silhouette_score(
    X_scaled[dbscan_non_noise],
    dbscan_labels[dbscan_non_noise],
)

hierarchical_silhouette = silhouette_score(
    X_scaled,
    hierarchical_labels,
)

comparison = pd.DataFrame(
    {
        "Method": ["K-Means", "DBSCAN", "Hierarchical (Ward)"],
        "Clusters": [
            len(np.unique(kmeans_labels)),
            n_dbscan_clusters,
            n_hierarchical_clusters,
        ],
        "Noise Points": [
            0,
            n_dbscan_noise,
            0,
        ],
        "Silhouette": [
            kmeans_silhouette,
            dbscan_silhouette,
            hierarchical_silhouette,
        ],
    }
)

comparison.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], c=kmeans_labels, s=22)
axes[0].set_title("K-Means")

axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=dbscan_labels, s=22)
axes[1].set_title("DBSCAN")

axes[2].scatter(X_scaled[:, 0], X_scaled[:, 1], c=hierarchical_labels, s=22)
axes[2].set_title("Hierarchical (Ward)")

for ax in axes:
    ax.set_xlabel("x1 (scaled)")
    ax.set_ylabel("x2 (scaled)")

plt.tight_layout()
plt.show()

## 10. Method Recommendation

For this dataset, **DBSCAN is the best fit**.

The reason is mainly the data shape:

- the two natural groups are curved and connected rather than compact around one centroid;
- several samples are isolated from the dense regions;
- DBSCAN can follow the non-convex structure and can explicitly label sparse points as noise.

K-Means is simpler and efficient, but its nearest-centroid partition cuts across the moon geometry and assigns every outlier to a cluster.

Hierarchical clustering adds useful information through the dendrogram and does not require fixing the final cluster count before building the hierarchy. With Ward linkage, however, the merges still favor compact variance-based groups, which is less suitable for this particular shape.

The method choice therefore depends on the structure of the data rather than on one algorithm being generally better than the others.

## 11. Final Notes

The main points from this experiment are:

- scaling matters whenever clustering depends on distance;
- K-Means is strongest when clusters are compact and can be represented by centroids;
- DBSCAN defines clusters through density using `eps` and `min_samples`;
- DBSCAN can separate noise from clusters, but its result can be sensitive to the density parameters;
- after an agglomerative merge, cluster distances are updated according to the selected linkage rule;
- a dendrogram shows the full merge hierarchy and can be cut at a selected height;
- clustering should be evaluated using both quantitative metrics and the actual geometry and meaning of the resulting groups.